# 51. The Classic Economic Order Quantity (EOQ) Problem

## Tier 1: The Pen & Paper Method (Mathematical Formulation)

### Key Assumptions
- Demand is constant and known deterministically
- Ordering cost is fixed per order regardless of order size
- Holding cost is linear with inventory level
- No stockouts allowed (infinite service level)
- Instantaneous replenishment (no lead time)
- Unit cost is constant regardless of order quantity

### Mathematical Approach
The Economic Order Quantity (EOQ) model balances two competing costs:
1. **Ordering Costs**: Costs incurred each time an order is placed
2. **Holding Costs**: Costs for maintaining inventory over time

The optimal order quantity occurs where the marginal ordering cost equals the marginal holding cost.

### What to Look For in Results
- Optimal order quantity that minimizes total annual cost
- Number of orders per year at optimal solution
- Order cycle length (days between orders)
- Cost breakdown showing balance between ordering and holding costs
- Sensitivity analysis showing how parameters affect optimal solution

In [1]:
# Import required libraries for mathematical analysis and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import warnings
warnings.filterwarnings('ignore')

# Set professional styling for plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [2]:
class EOQMathematicalModel:
    """
    Mathematical implementation of the Economic Order Quantity model.
    This class implements the classic EOQ formula with comprehensive analysis.
    """
    
    def __init__(self, demand_rate, ordering_cost, holding_cost, unit_cost=None):
        """
        Initialize EOQ model with problem parameters.
        
        Parameters:
        - demand_rate: Annual demand in units per year
        - ordering_cost: Fixed cost per order in dollars
        - holding_cost: Annual holding cost per unit in dollars
        - unit_cost: Cost per unit (optional, for total cost calculation)
        """
        self.D = demand_rate  # Annual demand rate
        self.S = ordering_cost  # Ordering cost per order
        self.H = holding_cost  # Holding cost per unit per year
        self.c = unit_cost  # Unit cost (optional)
        
    def calculate_optimal_eoq(self):
        """
        Calculate the optimal Economic Order Quantity using the classic formula.
        
        EOQ* = sqrt(2 * D * S / H)
        """
        if self.D <= 0 or self.S <= 0 or self.H <= 0:
            raise ValueError("All parameters must be positive")
            
        eoq_optimal = np.sqrt(2 * self.D * self.S / self.H)
        return eoq_optimal
    
    def calculate_associated_metrics(self, eoq):
        """
        Calculate all associated metrics for a given EOQ.
        """
        # Number of orders per year
        orders_per_year = self.D / eoq
        
        # Order cycle length in days
        days_between_orders = 365 / orders_per_year
        
        # Average inventory level (assuming constant depletion)
        avg_inventory = eoq / 2
        
        # Cost components
        annual_ordering_cost = orders_per_year * self.S
        annual_holding_cost = avg_inventory * self.H
        annual_inventory_cost = annual_ordering_cost + annual_holding_cost
        
        # Total cost including purchasing (if unit cost provided)
        total_annual_cost = annual_inventory_cost
        if self.c is not None:
            annual_purchasing_cost = self.D * self.c
            total_annual_cost += annual_purchasing_cost
        
        return {
            'eoq': eoq,
            'orders_per_year': orders_per_year,
            'days_between_orders': days_between_orders,
            'avg_inventory': avg_inventory,
            'annual_ordering_cost': annual_ordering_cost,
            'annual_holding_cost': annual_holding_cost,
            'annual_inventory_cost': annual_inventory_cost,
            'total_annual_cost': total_annual_cost
        }
    
    def sensitivity_analysis(self, parameter_variations=0.2):
        """
        Perform sensitivity analysis on EOQ with respect to all parameters.
        
        Parameters:
        - parameter_variations: Percentage variation to test (default 20%)
        """
        base_eoq = self.calculate_optimal_eoq()
        
        # Test variations in each parameter
        variations = np.linspace(1 - parameter_variations, 1 + parameter_variations, 5)
        
        results = {
            'demand_variations': [],
            'ordering_cost_variations': [],
            'holding_cost_variations': []
        }
        
        # Demand variations
        for var in variations:
            temp_model = EOQMathematicalModel(self.D * var, self.S, self.H, self.c)
            eoq_var = temp_model.calculate_optimal_eoq()
            results['demand_variations'].append(eoq_var)
        
        # Ordering cost variations
        for var in variations:
            temp_model = EOQMathematicalModel(self.D, self.S * var, self.H, self.c)
            eoq_var = temp_model.calculate_optimal_eoq()
            results['ordering_cost_variations'].append(eoq_var)
        
        # Holding cost variations
        for var in variations:
            temp_model = EOQMathematicalModel(self.D, self.S, self.H * var, self.c)
            eoq_var = temp_model.calculate_optimal_eoq()
            results['holding_cost_variations'].append(eoq_var)
        
        return results, variations

In [3]:
# Create visualization functions for EOQ analysis

def plot_eoq_cost_components(model, eoq_range=None):
    """
    Plot the cost components showing the EOQ optimization.
    """
    if eoq_range is None:
        base_eoq = model.calculate_optimal_eoq()
        eoq_range = np.linspace(base_eoq * 0.3, base_eoq * 2.5, 100)
    
    # Calculate costs for each EOQ value
    ordering_costs = []
    holding_costs = []
    total_costs = []
    
    for eoq in eoq_range:
        metrics = model.calculate_associated_metrics(eoq)
        ordering_costs.append(metrics['annual_ordering_cost'])
        holding_costs.append(metrics['annual_holding_cost'])
        total_costs.append(metrics['annual_inventory_cost'])
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 8))
    
    ax.plot(eoq_range, ordering_costs, 'b-', linewidth=2.5, label='Ordering Cost', alpha=0.8)
    ax.plot(eoq_range, holding_costs, 'r-', linewidth=2.5, label='Holding Cost', alpha=0.8)
    ax.plot(eoq_range, total_costs, 'g-', linewidth=3, label='Total Cost', alpha=0.9)
    
    # Mark optimal point
    optimal_eoq = model.calculate_optimal_eoq()
    optimal_metrics = model.calculate_associated_metrics(optimal_eoq)
    
    ax.plot(optimal_eoq, optimal_metrics['annual_inventory_cost'], 'ko', 
            markersize=10, label=f'Optimal EOQ: {optimal_eoq:.1f}')
    
    # Add vertical line at optimal EOQ
    ax.axvline(x=optimal_eoq, color='black', linestyle='--', alpha=0.5)
    
    # Formatting
    ax.set_xlabel('Order Quantity (units)', fontsize=14, fontweight='bold')
    ax.set_ylabel('Annual Cost ($)', fontsize=14, fontweight='bold')
    ax.set_title('EOQ Cost Components Analysis', fontsize=16, fontweight='bold', pad=20)
    ax.legend(fontsize=12, loc='best')
    ax.grid(True, alpha=0.3)
    
    # Format y-axis as currency
    ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f'${x:,.0f}'))
    
    plt.tight_layout()
    plt.show()
    
    return fig, ax

def plot_sensitivity_analysis(results, variations, model):
    """
    Plot sensitivity analysis results.
    """
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))
    
    # Demand variations
    ax1.plot(variations, results['demand_variations'], 'bo-', linewidth=2.5, markersize=8)
    ax1.set_xlabel('Demand Multiplier', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Optimal EOQ (units)', fontsize=12, fontweight='bold')
    ax1.set_title('EOQ Sensitivity to Demand', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=model.calculate_optimal_eoq(), color='red', linestyle='--', alpha=0.7, label='Base EOQ')
    ax1.legend()
    
    # Ordering cost variations
    ax2.plot(variations, results['ordering_cost_variations'], 'go-', linewidth=2.5, markersize=8)
    ax2.set_xlabel('Ordering Cost Multiplier', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Optimal EOQ (units)', fontsize=12, fontweight='bold')
    ax2.set_title('EOQ Sensitivity to Ordering Cost', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=model.calculate_optimal_eoq(), color='red', linestyle='--', alpha=0.7, label='Base EOQ')
    ax2.legend()
    
    # Holding cost variations
    ax3.plot(variations, results['holding_cost_variations'], 'mo-', linewidth=2.5, markersize=8)
    ax3.set_xlabel('Holding Cost Multiplier', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Optimal EOQ (units)', fontsize=12, fontweight='bold')
    ax3.set_title('EOQ Sensitivity to Holding Cost', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=model.calculate_optimal_eoq(), color='red', linestyle='--', alpha=0.7, label='Base EOQ')
    ax3.legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig

### Concrete Example - MegaMart Coffee Beans

Let's apply the mathematical EOQ model to MegaMart's Colombian Supremo coffee bean inventory problem:

**Problem Parameters:**
- Annual demand (D): 12,000 bags per year
- Ordering cost (S): $150 per order
- Holding cost (H): $8 per bag per year
- Unit cost: Not specified (focus on inventory costs)

In [ ]:
# Initialize the EOQ model with MegaMart's parameters
megamart_model = EOQMathematicalModel(
    demand_rate=12000,    # 12,000 bags per year
    ordering_cost=150,     # $150 per order
    holding_cost=8        # $8 per bag per year
)

# Calculate optimal EOQ
optimal_eoq = megamart_model.calculate_optimal_eoq()
print(f"Optimal EOQ: {optimal_eoq:.2f} bags")

# Calculate all associated metrics
metrics = megamart_model.calculate_associated_metrics(optimal_eoq)

print("\n=== MegaMart Coffee Bean EOQ Analysis ===")
print(f"Optimal Order Quantity: {metrics['eoq']:.2f} bags")
print(f"Orders per Year: {metrics['orders_per_year']:.2f}")
print(f"Days Between Orders: {metrics['days_between_orders']:.1f}")
print(f"Average Inventory Level: {metrics['avg_inventory']:.2f} bags")
print(f"Annual Ordering Cost: ${metrics['annual_ordering_cost']:,.2f}")
print(f"Annual Holding Cost: ${metrics['annual_holding_cost']:,.2f}")
print(f"Annual Inventory Cost: ${metrics['annual_inventory_cost']:,.2f}")

# Verify the balance between ordering and holding costs at optimal EOQ
cost_balance = abs(metrics['annual_ordering_cost'] - metrics['annual_holding_cost'])
print(f"\nCost Balance (|Ordering - Holding|): ${cost_balance:.2f}")
print(f"Balance Percentage: {(cost_balance/metrics['annual_inventory_cost'])*100:.3f}%")

In [ ]:
# Create comprehensive cost visualization
fig, ax = plot_eoq_cost_components(megamart_model)
plt.show()

In [ ]:
# Perform sensitivity analysis
sensitivity_results, variations = megamart_model.sensitivity_analysis(parameter_variations=0.2)

print("\n=== Sensitivity Analysis Results ===")
print("Parameter Variation | Demand EOQ | Order Cost EOQ | Hold Cost EOQ")
print("-" * 65)

for i, var in enumerate(variations):
    demand_eoq = sensitivity_results['demand_variations'][i]
    order_eoq = sensitivity_results['ordering_cost_variations'][i]
    hold_eoq = sensitivity_results['holding_cost_variations'][i]
    print(f"{var:>17.2f} | {demand_eoq:>9.1f} | {order_eoq:>12.1f} | {hold_eoq:>12.1f}")

# Plot sensitivity analysis
fig = plot_sensitivity_analysis(sensitivity_results, variations, megamart_model)
plt.show()

In [ ]:
# Generate order schedule for the first quarter
def generate_order_schedule(model, days=90):
    """
    Generate an order schedule for a specified period.
    """
    optimal_eoq = model.calculate_optimal_eoq()
    metrics = model.calculate_associated_metrics(optimal_eoq)
    days_between_orders = metrics['days_between_orders']
    
    schedule = []
    current_day = 1
    
    while current_day <= days:
        schedule.append({
            'day': int(current_day),
            'order_quantity': optimal_eoq
        })
        current_day += days_between_orders
    
    return schedule

# Generate first quarter schedule
schedule = generate_order_schedule(megamart_model, days=90)

print("\n=== First Quarter Order Schedule ===")
for order in schedule:
    print(f"Day {order['day']}: Order {order['order_quantity']:.2f} bags")

### Mathematical Verification and Insights

The mathematical EOQ solution demonstrates several key insights:

1. **Cost Balance**: At the optimal EOQ, ordering costs equal holding costs
2. **Square Root Relationship**: EOQ follows a square root law with respect to parameters
3. **Parameter Sensitivity**: EOQ is more sensitive to changes in holding costs than ordering costs
4. **Economies of Scale**: Higher demand leads to proportionally smaller increases in EOQ

### Why This Mathematical Approach Matters

The pen-and-paper mathematical formulation provides:
- **Theoretical Foundation**: Understanding of the fundamental trade-offs
- **Closed-Form Solution**: Immediate calculation without iterative methods
- **Parameter Insights**: Clear understanding of how each parameter affects the solution
- **Benchmark**: Baseline for comparing more complex methods

### Limitations of the Mathematical Approach

While elegant, the mathematical EOQ model has limitations:
- Assumes deterministic demand (no uncertainty)
- Ignores lead times and safety stock requirements
- No quantity discounts or price breaks
- No capacity constraints or storage limitations
- Single product focus (no multi-product interactions)

These limitations motivate the need for more advanced computational approaches in subsequent tiers.